# Make a Basic Neural Network by using PyTorch

# Code Flow -->

1. Load the Dataset

2. Basic Preprocessing

3. Training Process

-------a. Create the Model

-------b. Forward Pass

-------c. Loss Calculation

-------d. Back Propagation

-------e. Parameters/Weight Update

4. Model Evaluation

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [4]:
df.shape

(569, 33)

In [5]:
df.drop(columns=['id', 'Unnamed: 32'],inplace=True)

In [6]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


# Train-Test-Split

In [7]:
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling

In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

In [9]:
y_train

68     B
181    M
63     B
248    B
60     B
      ..
71     B
106    B
270    B
435    M
102    B
Name: diagnosis, Length: 455, dtype: str

# To remove alphabets need to do encoding on Y

In [10]:
encoder = LabelEncoder()
y_train_encoded = encoder.fit_transform(y_train)
y_test_encoded = encoder.transform(y_test)

# Numpy Arrays to PyTorch Conversion

In [11]:
X_train_tensor = torch.from_numpy(X_train_scaled)
X_test_tensor = torch.from_numpy(X_test_scaled)
y_train_tensor = torch.from_numpy(y_train_encoded)
y_test_tensor = torch.from_numpy(y_test_encoded)

In [12]:
X_train_tensor.shape, y_train_tensor.shape

(torch.Size([455, 30]), torch.Size([455]))

# Defining the Model

In [40]:
class mySimpleNN(nn.Module):

    def __init__(self, num_features):
        super(mySimpleNN, self).__init__()
        self.linear1 = nn.Linear(num_features, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, features):
        x_out = self.linear1(features)
        x_out = self.sigmoid(x_out)
        return x_out


# Important Parameters

In [25]:
learning_rate = 0.1
epochs = 15

In [36]:
# Define Loss Function:

loss_function = nn.BCELoss()  # Binary Cross-Entropy Loss

# Training Pipeline

In [37]:
# Create Model:

model = mySimpleNN(X_train_tensor.shape[1])

# Optimizer:

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# Define Loop:
for epoch in range(epochs):

    # Forward Pass:
    X_train_tensor = X_train_tensor.float()
    y_train_tensor = y_train_tensor.float().view(-1, 1)

    y_pred = model(X_train_tensor)

    # Loss Function:
    loss = loss_function(y_pred, y_train_tensor)

    # Clear Gradients:
    optimizer.zero_grad()

    # Backward Pass:
    loss.backward()

    # Update Parameters:
    optimizer.step()

    # Print the loss every epoch
    print(f'Epoch {epoch+1}, Loss: {loss.item()}')

Epoch 1, Loss: 0.67093825340271
Epoch 2, Loss: 0.5101734399795532
Epoch 3, Loss: 0.42159023880958557
Epoch 4, Loss: 0.36691468954086304
Epoch 5, Loss: 0.3299083709716797
Epoch 6, Loss: 0.3031053841114044
Epoch 7, Loss: 0.2826927900314331
Epoch 8, Loss: 0.2665422260761261
Epoch 9, Loss: 0.25337785482406616
Epoch 10, Loss: 0.24239006638526917
Epoch 11, Loss: 0.233040913939476
Epoch 12, Loss: 0.22495882213115692
Epoch 13, Loss: 0.2178790122270584
Epoch 14, Loss: 0.21160739660263062
Epoch 15, Loss: 0.20599845051765442


In [38]:
model.linear1.weight

Parameter containing:
tensor([[ 2.2162e-02,  1.0106e-01,  2.4020e-01,  1.9469e-01,  1.0590e-01,
          2.1274e-02,  1.5034e-01,  1.4104e-01,  1.7698e-01, -6.2164e-02,
          1.5806e-01, -5.8862e-02,  1.9824e-01,  2.7150e-01,  1.4144e-04,
          1.8207e-01, -5.9999e-02, -9.3859e-02,  1.4399e-01, -1.7050e-01,
          9.1750e-02,  1.9299e-01,  2.9288e-01,  3.5082e-01,  1.4038e-01,
          2.1776e-01,  1.1388e-01,  3.4610e-01,  1.2032e-02, -3.6646e-02]],
       requires_grad=True)

# Evaluation

In [39]:
with torch.no_grad():
    y_pred_test = model(X_test_tensor.float())
    y_pred_test_class = (y_pred_test >= 0.5).float()  # Convert probabilities to binary class labels

    # Calculate accuracy
    accuracy = (y_pred_test_class.squeeze() == y_test_tensor).float().mean()
    print(f'Test Accuracy: {accuracy.item()}')

Test Accuracy: 0.9824561476707458
